---
# Fase 10 — Ajuste de Hiperparâmetros (Tuning)

**Objetivo:** Otimizar os hiperparâmetros dos dois modelos mais promissores selecionados na Fase 9:
1. **Regressão Logística** (com `class_weight='balanced'`)
2. **Random Forest** (com `class_weight='balanced'`)

Utilizaremos o `GridSearchCV` para a Regressão Logística e o `RandomizedSearchCV` para o Random Forest para encontrar as configurações que maximizam o **Recall**.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# Carregar os dados de treino
X_train = pd.read_csv('data/X_train.csv')
y_train = pd.read_csv('data/y_train.csv')['Dataset']
colunas_numericas = list(X_train.columns)

# Importações para o Pipeline e Busca
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV

# Funções base para construção das pipelines
def criar_pipeline_lr():
    pre = ColumnTransformer(transformers=[
        ('num', ImbPipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), colunas_numericas)
    ])
    from sklearn.linear_model import LogisticRegression
    # class_weight='balanced' foi a estratégia vencedora
    return ImbPipeline(steps=[
        ('preprocessor', pre),
        ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))
    ])

def criar_pipeline_rf():
    # Random Forest não precisa de Scaler
    pre = ColumnTransformer(transformers=[
        ('num', ImbPipeline(steps=[
            ('imputer', SimpleImputer(strategy='median'))
        ]), colunas_numericas)
    ])
    from sklearn.ensemble import RandomForestClassifier
    return ImbPipeline(steps=[
        ('preprocessor', pre),
        ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Dados carregados e Pipelines base criadas.')

Dados carregados e Pipelines base criadas.


## 10.1 — Tuning: Regressão Logística

A Regressão Logística possui poucos hiperparâmetros, então podemos usar uma busca em grade (`GridSearchCV`) exaustiva. 
Queremos testar a força da regularização (`C`) e o tipo de penalidade.

In [2]:
# 1. Pipeline Base
pipe_lr = criar_pipeline_lr()

# 2. Espaço de Hiperparâmetros
param_grid_lr = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear', 'saga'] # Ambos suportam l1 e l2
}

# 3. Configurar GridSearchCV
grid_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    cv=cv,
    scoring='recall', # Otimizando para identificar mais pacientes doentes
    n_jobs=-1,
    verbose=1
)

# 4. Executar Busca
print('Buscando melhores hiperparâmetros para Regressão Logística...')
grid_lr.fit(X_train, y_train)

print(f'\nMelhor Recall médio no CV: {grid_lr.best_score_:.4f}')
print('Melhores hiperparâmetros encontrados:')
for param, valor in grid_lr.best_params_.items():
    print(f" - {param.replace('classifier__', '')}: {valor}")

Buscando melhores hiperparâmetros para Regressão Logística...
Fitting 5 folds for each of 24 candidates, totalling 120 fits



Melhor Recall médio no CV: 0.5618
Melhores hiperparâmetros encontrados:
 - C: 100
 - penalty: l1
 - solver: liblinear


## 10.2 — Tuning: Random Forest

O Random Forest tem um espaço de busca enorme. Para economizar poder computacional e tempo sem perder a chance de achar um excelente modelo, usaremos a busca aleatória (`RandomizedSearchCV`).

In [3]:
# 1. Pipeline Base
pipe_rf = criar_pipeline_rf()

# 2. Espaço de Hiperparâmetros
param_dist_rf = {
    'classifier__n_estimators': [100, 200, 300, 500],
    'classifier__max_depth': [None, 5, 10, 15, 20, 25],
    'classifier__min_samples_split': [2, 5, 10, 15],
    'classifier__min_samples_leaf': [1, 2, 4, 8],
    'classifier__max_features': ['sqrt', 'log2', None]
}

# 3. Configurar RandomizedSearchCV
rand_rf = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=param_dist_rf,
    n_iter=50, # Testa 50 combinações aleatórias
    cv=cv,
    scoring='recall', # Otimizando para Recall
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 4. Executar Busca
print('Buscando melhores hiperparâmetros para Random Forest...')
rand_rf.fit(X_train, y_train)

print(f'\nMelhor Recall médio no CV: {rand_rf.best_score_:.4f}')
print('Melhores hiperparâmetros encontrados:')
for param, valor in rand_rf.best_params_.items():
    print(f" - {param.replace('classifier__', '')}: {valor}")

Buscando melhores hiperparâmetros para Random Forest...
Fitting 5 folds for each of 50 candidates, totalling 250 fits



Melhor Recall médio no CV: 0.8830
Melhores hiperparâmetros encontrados:
 - n_estimators: 100
 - min_samples_split: 2
 - min_samples_leaf: 1
 - max_features: None
 - max_depth: 20


## 10.3 — Salvando os Modelos Otimizados

Agora que encontramos as melhores configurações para os dois algoritmos, vamos salvar as pipelines completas otimizadas na nossa pasta `models/`. Na próxima fase (Avaliação Final), finalmente tocaremos nos dados de Teste para ver qual desses dois "campeões" se sai melhor no mundo real.

In [4]:
import os
os.makedirs('models', exist_ok=True)

# Salvar o melhor modelo de Regressão Logística
melhor_lr = grid_lr.best_estimator_
joblib.dump(melhor_lr, 'models/best_lr_tuned.joblib')

# Salvar o melhor modelo Random Forest
melhor_rf = rand_rf.best_estimator_
joblib.dump(melhor_rf, 'models/best_rf_tuned.joblib')

print('Modelos tunados salvos com sucesso na pasta models/ !')
print('- models/best_lr_tuned.joblib')
print('- models/best_rf_tuned.joblib')

Modelos tunados salvos com sucesso na pasta models/ !
- models/best_lr_tuned.joblib
- models/best_rf_tuned.joblib


### Checklist de Conclusão da Fase 10
- ✅ Grelha de hiperparâmetros definida para LR e RF.
- ✅ `GridSearchCV` executado na Regressão Logística otimizando o *Recall*.
- ✅ `RandomizedSearchCV` executado no Random Forest otimizando o *Recall*.
- ✅ Melhores pipelines exportadas e salvas (`.joblib`).
- ✅ Prontos para a Avaliação Final contra o conjunto de Teste.